# P2. A mean that depends on an input
Book: Linear Regression; Pattern Matching, Model Estimation.

Supplement: Alisa's math notes, [Derivative rules](https://alisawuffles.notion.site/math-notes#3737eb873605801e8c0aee1e484960c7), and Alisa's book of LLMs, [Gradients](https://alisawuffles.notion.site/alisa-s-book-of-llms#2de7eb87360580dcbb05cbac4516f90b). Use the local-sensitivity explanation and our finite-difference check.
These selected readings are optional support. The classroom examples define the required scope.
Reading caution: finite differences approximate derivatives at a nonzero step size.

Use J = sum of squared residuals / (2n) for the gradient exercise.

## Setup
Run this cell once. Helpers support the experiments below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 12})

def regression_data(seed=600, n=100):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-2, 2, n)
    y = 1 + 2*x + 0.7*x*x + rng.normal(0, 1, n)
    return x.reshape(-1, 1), y

## Warmup inside the worked explanation
Conditional means are 2 and 6; group probabilities are .25 and .75.
Predict the overall mean. Why is an unweighted average inappropriate?

In [ ]:
group_probability = np.array([.25, .75])
conditional_mean = np.array([2., 6.])
print("Total expectation:", group_probability @ conditional_mean)

## A. One update by hand (25 minutes, including shape and derivative checks)
Predict the signs of the intercept and slope gradients at zero.
Calculate one update using learning rate 0.1, then run the check.

In [ ]:
x_small = np.array([0., 1., 2.])
y_small = np.array([1., 3., 3.])
X_small = np.c_[np.ones(3), x_small]
theta = np.array([0., 0.])
learning_rate = .1
def loss(theta):
    return np.mean((X_small @ theta-y_small)**2)/2
gradient = X_small.T @ (X_small @ theta-y_small)/len(y_small)
next_theta = theta-learning_rate*gradient
print("Gradient:", gradient)
print("Next parameters:", next_theta)
print("Loss before and after:", loss(theta), loss(next_theta))
assert np.allclose(gradient, [-7/3, -3])

Before running: predict the shape of each expression. Does theta.T make a column?

In [ ]:
for name, array in [("X @ theta", X_small @ theta), ("X * theta", X_small * theta),
                    ("X.T", X_small.T), ("theta.T", theta.T),
                    ("theta[:, None]", theta[:, None])]:
    print(name, array.shape)
epsilon = 1e-5
directions = np.eye(len(theta))
numerical_gradient = np.array([(loss(theta+epsilon*e)-loss(theta-epsilon*e))/(2*epsilon)
                               for e in directions])
print("Analytic and finite-difference gradients:", gradient, numerical_gradient)
assert np.allclose(gradient, numerical_gradient, atol=1e-8)

Set the learning rate to zero, then to 2. Predict before each run.
Does subtracting the gradient guarantee improvement for every step size?

Prediction:

Observation:

Explanation:

## B. The residuals tell a story (25 minutes)
Fit a straight line to the common simulated regression data.
Predict which structure will remain in the residuals.

In [ ]:
x, y = regression_data()
X = np.c_[np.ones(len(y)), x[:, 0]]
coef = np.linalg.lstsq(X, y, rcond=None)[0]
fitted = X @ coef
grid = np.linspace(-2, 2, 200)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(x[:, 0], y, alpha=.6)
axes[0].plot(grid, coef[0]+coef[1]*grid, color="black")
axes[0].set(xlabel="x", ylabel="y", title="Fitted line")
axes[1].scatter(x[:, 0], y-fitted, alpha=.6)
axes[1].axhline(0, color="black")
axes[1].set(xlabel="x", ylabel="Observed minus fitted", title="Residuals")
plt.tight_layout()
plt.show()

Insert `y[0] += 15` immediately after generating the data, then rerun that cell.
Explain the influence of that point. Regenerating the data prevents repeated additions.
Then restore the original data. Identify a limitation of the straight-line model.

Prediction:

Observation:

Explanation:

## Individual exit
Explain a slope in units, a curved residual pattern, and why a small training
loss alone cannot tell us how the model will perform on new observations.